# Laboratorio 7: Spark MLlib

## Inciso 6. Pipeline de Random Forest

**Equipo:** Abby Donis (22440), Hansel Lopez (19026), Fabian Prado (23427)

**Datos.** Igual que en el inciso 5, se lee `processed/eneic_2025.parquet` (asalariados de 2025 ya filtrados). Train: 2025T1-T3, validación: 2025T4.

**Tarea.** Estimar `salario_mensual` con los mismos seis predictores: `edad`, `antiguedad`, `horas_semanales`, `nivel_educativo`, `categoria_ocupacional` y `dominio`. La diferencia con la regresión lineal es que Random Forest no requiere estandarización y puede capturar interacciones y no linealidades.

In [1]:
import json
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd()))

import matplotlib.pyplot as plt
import pandas as pd
from pyspark.ml import PipelineModel
from pyspark.ml.regression import RandomForestRegressor
from pyspark.sql import functions as F

from src import config, datos, modelos

config.preparar_directorios()
pd.set_option("display.float_format", "{:,.3f}".format)

spark = datos.iniciar_spark()
df = datos.cargar(spark, 2025)

### 6.1 Entrenamiento y validación

Misma división temporal que el inciso 5: 2025T1-T3 para entrenar, 2025T4 para validar. Así la comparación es directa.

In [2]:
train, valid = modelos.dividir(df)
train, valid = train.cache(), valid.cache()

print("Entrenamiento (2025T1-T3):", train.count())
print("Validación (2025T4):", valid.count())

Entrenamiento (2025T1-T3): 40361
Validación (2025T4): 12664


### 6.2 Modelo de referencia

Mismo modelo de referencia que el inciso 5: predecir la media del salario de entrenamiento para todos. Sirve para comparar.

In [3]:
pred_ref, media_train = modelos.predecir_referencia(train, valid)
metricas_ref = modelos.metricas(pred_ref)
print(f"Salario medio de entrenamiento: Q{media_train:,.2f}")
pd.DataFrame([metricas_ref], index=["referencia"])

Salario medio de entrenamiento: Q3,383.12


mae      rmse     r2
referencia 1,672.503 2,889.571 -0.003

### 6.3 Pipeline

El pipeline usa las mismas etapas de codificación que la regresión lineal:

a. **Categóricas.** `StringIndexer` + `OneHotEncoder` para `nivel_educativo`, `categoria_ocupacional` y `dominio`.

b. **VectorAssembler.** Junta las tres numéricas (`edad`, `antiguedad`, `horas_semanales`) con las categóricas codificadas en un vector `features`.

c. **Sin estandarización.** Random Forest no necesita escalar los predictores porque divide los datos en cada nodo, no calcula distancias.

d. **Modelo.** `RandomForestRegressor` sobre `salario_mensual`.

**Hiperparámetros.** Se varía `numTrees` (cantidad de árboles) y `maxDepth` (profundidad máxima). Más árboles reduce el sobreajuste pero tarda más. Profundidad mayor captura interacciones más complejas pero puede sobreajustar. Se prueba una grilla con `numTrees` en [50, 100, 200] y `maxDepth` en [5, 10, 15, None], donde None significa sin límite. Semilla fija para reproducibilidad.

In [4]:
ejemplo = modelos.pipeline_regresion(RandomForestRegressor(featuresCol="features", labelCol=config.OBJETIVO))
for etapa in ejemplo.getStages():
    print(type(etapa).__name__)

StringIndexer
OneHotEncoder
VectorAssembler
RandomForestRegressor


In [5]:
NUM_TREES = [50, 100, 200]
MAX_DEPTHS = [5, 10, 15, None]
configuraciones = [
    {"numTrees": t, "maxDepth": d} for t in NUM_TREES for d in MAX_DEPTHS
]


def construir_rf(params):
    """RandomForestRegressor con la semilla fija y los hiperparámetros indicados."""
    return RandomForestRegressor(featuresCol="features", labelCol=config.OBJETIVO,
                                 seed=config.SEMILLA, **params)


grilla, ajustados = modelos.ajustar_grilla(train, valid, configuraciones, construir_rf)

In [6]:
grilla_ordenada = grilla.sort_values("rmse_valid")
grilla_ordenada

**Lectura de la grilla.**

- La mejor configuración tiene `numTrees = 200` y `maxDepth = 15` con RMSE de validación de Q1,987. Esto es mejor que la regresión lineal (Q2,186).
- Más árboles ayuda: 200 árboles supera a 100 y 50 en casi todas las profundidades.
- Profundidad 15 es la mejor, seguida de 10. Profundidad 5 queda corta (subajuste) y None (sin límite) no mejora, posiblemente por sobreajuste.
- La combinación ganadora captura interacciones complejas sin sobreajustar demasiado.

In [7]:
mejor = grilla.loc[grilla["rmse_valid"].idxmin()]
MEJOR_PARAMS = {"numTrees": int(mejor["numTrees"]), "maxDepth": int(mejor["maxDepth"]) if mejor["maxDepth"] is not None else None}
modelo_final = ajustados[int(mejor["config"])]
pred_valid = modelo_final.transform(valid).cache()
metricas_rf = modelos.metricas(pred_valid)

print("Configuración elegida:", MEJOR_PARAMS)

Configuración elegida: {'numTrees': 200, 'maxDepth': 15}


In [8]:
lineal_json = json.loads((config.MODELOS / "regresion_lineal.json").read_text())
metricas_lineal = lineal_json["metricas_validacion"]

comparacion = pd.DataFrame([metricas_ref, metricas_lineal, metricas_rf],
                           index=["referencia", "regresión lineal", "random forest"])
comparacion.loc["mejora vs ref (%)"] = [
    float("nan"),
    100 * (1 - metricas_lineal["rmse"] / metricas_ref["rmse"]),
    100 * (1 - metricas_rf["rmse"] / metricas_ref["rmse"]),
]
comparacion

**Comparación.** Random Forest baja el RMSE a Q1,987 (31.3% mejor que la referencia), mientras que la regresión lineal quedó en Q2,186 (24.3% mejor). El R2 sube a 0.523, explicando el 52% de la variación del salario, vs 43% del lineal.

Random Forest gana porque captura interacciones: el efecto de la educación puede ser distinto según la categoría ocupacional, y el modelo lineal no puede representar eso. Además, no asume linealidad aditiva.

### 6.4 Importancia de variables

Random Forest calcula la importancia de cada predictor midiendo cuánto reduce el error al dividir por esa variable. Se normaliza para que sumen 1.

In [9]:
rf = modelo_final.stages[-1]
importancias = pd.DataFrame({"variable": modelos.nombres_features(modelo_final),
                             "importancia": list(rf.featureImportances)})


def etiqueta(nombre):
    """Traduce 'nivel_educativo=5' a 'nivel_educativo: Superior'."""
    if "=" not in nombre:
        return nombre
    columna, codigo = nombre.split("=")
    return f"{columna}: {config.ETIQUETAS[columna].get(codigo, codigo)}"


importancias["variable"] = importancias["variable"].map(etiqueta)
importancias = importancias.sort_values("importancia", ascending=False)
importancias

In [10]:
fig, eje = plt.subplots(figsize=(10, 6))
top = importancias.head(15)
eje.barh(top["variable"][::-1], top["importancia"][::-1], color=config.COLORES["principal"])
eje.set_xlabel("Importancia")
eje.set_title("Top 15 predictores más importantes (Random Forest)")
eje.grid(axis="x", alpha=0.3)
fig.tight_layout()
fig.savefig(config.FIGURAS / "06_importancias.png", dpi=120)
plt.show()

**Lectura.** Las variables más importantes son:

1. **nivel_educativo: Doctorado** y **Maestría** - el nivel educativo es lo que más pesa
2. **nivel_educativo: Superior** - también educación
3. **categoria_ocupacional: Gobierno** - trabajar en gobierno paga más
4. **edad** y **antiguedad** - variables numéricas importantes

La educación domina el top, confirmando lo visto en la regresión lineal. Pero ahora las categóricas aparecen con más peso relativo porque el modelo puede capturar sus interacciones con las numéricas.

### 6.5 Guardar el mejor modelo

Se guarda el `PipelineModel` completo en `modelos/random_forest`, junto con un JSON con la configuración elegida y sus métricas.

In [11]:
ruta_modelo = config.MODELOS / "random_forest"
modelos.guardar_modelo(modelo_final, ruta_modelo)

resumen = {
    "modelo": "RandomForestRegressor",
    "entrenamiento": modelos.PERIODOS_TRAIN,
    "validacion": modelos.PERIODO_VALID,
    "parametros": {**MEJOR_PARAMS, "seed": config.SEMILLA},
    "metricas_validacion": metricas_rf,
    "metricas_referencia": metricas_ref,
}
(config.MODELOS / "random_forest.json").write_text(json.dumps(resumen, indent=2))

cargado = PipelineModel.load(str(ruta_modelo))
print("RMSE del modelo cargado:", round(modelos.metricas(cargado.transform(valid))["rmse"], 2))

RMSE del modelo cargado: 1987.42


### 6.6 Conclusión del inciso 6

- Random Forest con 200 árboles y profundidad 15 obtuvo RMSE de Q1,987, mejor que la regresión lineal (Q2,186).
- El R2 sube a 0.523, explicando el 52% de la variación del salario vs 43% del lineal.
- Las variables más importantes son las educativas (doctorado, maestría, superior) y la categoría ocupacional (gobierno).
- Random Forest gana porque captura interacciones y no linealidades que el modelo lineal no puede.
- El mejor modelo queda guardado en `modelos/random_forest` y su configuración en `modelos/random_forest.json`.

In [12]:
spark.stop()